# Task 1 — Query Class Validation

Validates the paraphrase and entity query classes built in Steps 2–3.

**What to do:**
1. Run all cells.
2. In the **Manual Rating** sections, update the `ratings` dicts with your 1–5 scores.
3. Re-run the summary cells to get final statistics.
4. Copy the printed summary into `results/task1_class_validation.md`.

In [ ]:
import json
import random
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA_DIR = Path('/home/ishana/scratch/data/classes')

with open(DATA_DIR / 'paraphrase_classes.json') as f:
    para_classes = json.load(f)

with open(DATA_DIR / 'entity_classes.json') as f:
    entity_classes = json.load(f)

para_embs = np.load(DATA_DIR / 'paraphrase_embeddings.npy')   # (100, 6, 1024)
entity_embs = np.load(DATA_DIR / 'entity_embeddings.npy')      # (20, max_size, 1024)

print(f'Paraphrase classes: {len(para_classes)}')
print(f'Entity classes:     {len(entity_classes)}')
print(f'Para embeddings:    {para_embs.shape}')
print(f'Entity embeddings:  {entity_embs.shape}')

---
## Part 1 — Paraphrase Classes

Showing 20 random classes. Rate each 1–5:
- **5** = All paraphrases clearly ask the exact same thing
- **4** = Mostly good, one minor deviation
- **3** = Acceptable but noticeable differences
- **2** = Several paraphrases drift in meaning
- **1** = Not paraphrases at all

In [ ]:
random.seed(42)
sample_indices = sorted(random.sample(range(len(para_classes)), 20))
sample_para = [(i, para_classes[i]) for i in sample_indices]

for rank, (idx, cls) in enumerate(sample_para, 1):
    sim = cls['within_class_similarity']
    print(f"[{rank:02d}] idx={idx}  sim={sim:.3f}")
    print(f"  Original: {cls['original_query']}")
    for j, p in enumerate(cls['paraphrases'], 1):
        print(f"  {j}. {p}")
    print()

In [ ]:
# ── FILL IN YOUR RATINGS HERE (1–5 per class, in display order) ──────────────
# Keys are the rank numbers shown above (1–20).
# Replace None with your rating after reviewing each class.

para_ratings = {
     1: None,
     2: None,
     3: None,
     4: None,
     5: None,
     6: None,
     7: None,
     8: None,
     9: None,
    10: None,
    11: None,
    12: None,
    13: None,
    14: None,
    15: None,
    16: None,
    17: None,
    18: None,
    19: None,
    20: None,
}

In [ ]:
# Paraphrase rating summary
rated = {k: v for k, v in para_ratings.items() if v is not None}
if rated:
    scores = list(rated.values())
    print(f'Rated {len(rated)}/20 paraphrase classes')
    print(f'Mean rating:  {np.mean(scores):.2f} / 5.0')
    print(f'Median:       {np.median(scores):.1f}')
    print(f'Distribution: {dict(sorted((v, scores.count(v)) for v in set(scores)))}')
    print(f'Pass (≥4.0): {"YES" if np.mean(scores) >= 4.0 else "NO"}')
else:
    print('No ratings yet — fill in para_ratings above.')

---
## Part 2 — Entity Classes

Showing all 20 entity classes. Rate each 1–5:
- **5** = Clearly all about the same entity, coherent group
- **4** = Good grouping, minor entity ambiguity
- **3** = Acceptable but entity is too broad (e.g. 'China' mixes history/culture/geography)
- **2** = Some queries don't belong
- **1** = Incoherent grouping

In [ ]:
for rank, cls in enumerate(entity_classes, 1):
    sim = cls['within_class_similarity']
    print(f"[{rank:02d}] entity='{cls['primary_entity']}'  size={cls['size']}  sim={sim:.3f}")
    for q in cls['queries']:
        print(f"  - {q}")
    print()

In [ ]:
# ── FILL IN YOUR RATINGS HERE (1–5 per entity class) ─────────────────────────

entity_ratings = {
     1: None,  # China
     2: None,  # Congress
     3: None,  # Australia
     4: None,  # Nba
     5: None,  # Britain
     6: None,
     7: None,
     8: None,
     9: None,
    10: None,
    11: None,
    12: None,
    13: None,
    14: None,
    15: None,
    16: None,
    17: None,
    18: None,
    19: None,
    20: None,
}

In [ ]:
# Entity rating summary
rated_e = {k: v for k, v in entity_ratings.items() if v is not None}
if rated_e:
    scores_e = list(rated_e.values())
    print(f'Rated {len(rated_e)}/20 entity classes')
    print(f'Mean rating:  {np.mean(scores_e):.2f} / 5.0')
    print(f'Median:       {np.median(scores_e):.1f}')
    print(f'Distribution: {dict(sorted((v, scores_e.count(v)) for v in set(scores_e)))}')
    print(f'Pass (≥3.5): {"YES" if np.mean(scores_e) >= 3.5 else "NO"}')
else:
    print('No ratings yet — fill in entity_ratings above.')

---
## Part 3 — Within-Class Similarity Histograms

In [ ]:
para_sims = [c['within_class_similarity'] for c in para_classes]
entity_sims = [c['within_class_similarity'] for c in entity_classes]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(para_sims, bins=20, color='steelblue', edgecolor='white', linewidth=0.5)
axes[0].axvline(np.mean(para_sims), color='red', linestyle='--', linewidth=1.5, label=f'mean={np.mean(para_sims):.3f}')
axes[0].axvline(0.85, color='orange', linestyle=':', linewidth=1.5, label='target=0.85')
axes[0].set_title('Paraphrase Classes\nWithin-class Cosine Similarity', fontsize=12)
axes[0].set_xlabel('Mean Pairwise Cosine Similarity')
axes[0].set_ylabel('Count')
axes[0].legend()
axes[0].set_xlim(0.6, 1.0)

axes[1].hist(entity_sims, bins=10, color='coral', edgecolor='white', linewidth=0.5)
axes[1].axvline(np.mean(entity_sims), color='red', linestyle='--', linewidth=1.5, label=f'mean={np.mean(entity_sims):.3f}')
axes[1].set_title('Entity Classes\nWithin-class Cosine Similarity', fontsize=12)
axes[1].set_xlabel('Mean Pairwise Cosine Similarity')
axes[1].set_ylabel('Count')
axes[1].legend()
axes[1].set_xlim(0.3, 0.8)

plt.tight_layout()
plt.savefig(DATA_DIR / 'similarity_histograms.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to', DATA_DIR / 'similarity_histograms.png')

---
## Part 4 — Entity Class Size Distribution

In [ ]:
sizes = [c['size'] for c in entity_classes]
from collections import Counter
size_dist = Counter(sizes)

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(size_dist.keys(), size_dist.values(), color='coral', edgecolor='white')
ax.set_title('Entity Class Size Distribution', fontsize=12)
ax.set_xlabel('Queries per Class')
ax.set_ylabel('Number of Classes')
ax.set_xticks(sorted(size_dist.keys()))
plt.tight_layout()
plt.savefig(DATA_DIR / 'entity_size_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print('Entity class size distribution:')
for sz in sorted(size_dist):
    print(f'  size {sz}: {size_dist[sz]} classes')
print(f'Total queries: {sum(sizes)}')
print(f'Mean size: {np.mean(sizes):.1f}')

---
## Part 5 — Full Summary (copy to results/task1_class_validation.md)

In [ ]:
rated_p = {k: v for k, v in para_ratings.items() if v is not None}
rated_e = {k: v for k, v in entity_ratings.items() if v is not None}

print('=' * 60)
print('TASK 1 — QUERY CLASS VALIDATION SUMMARY')
print('=' * 60)
print()
print('PARAPHRASE CLASSES')
print(f'  Count:                  {len(para_classes)}')
print(f'  Queries per class:      6 (1 original + 5 paraphrases)')
print(f'  Model used:             Mistral-7B-Instruct-v0.2')
print(f'  Embedding model:        BAAI/bge-large-en-v1.5')
print(f'  Within-class sim mean:  {np.mean(para_sims):.3f}')
print(f'  Within-class sim min:   {np.min(para_sims):.3f}')
print(f'  Within-class sim max:   {np.max(para_sims):.3f}')
if rated_p:
    scores_p = list(rated_p.values())
    print(f'  Manual rating (n={len(rated_p)}):   {np.mean(scores_p):.2f} / 5.0  (pass ≥4.0: {"YES" if np.mean(scores_p) >= 4.0 else "NO"})')
else:
    print(f'  Manual rating:          NOT YET RATED')
print()
print('ENTITY CLASSES')
print(f'  Count:                  {len(entity_classes)}')
sizes = [c["size"] for c in entity_classes]
print(f'  Class size range:       {min(sizes)}–{max(sizes)}  (mean {np.mean(sizes):.1f})')
print(f'  Total queries:          {sum(sizes)}')
print(f'  Embedding model:        BAAI/bge-large-en-v1.5')
print(f'  Within-class sim mean:  {np.mean(entity_sims):.3f}  (expected 0.5–0.7)')
print(f'  Within-class sim min:   {np.min(entity_sims):.3f}')
print(f'  Within-class sim max:   {np.max(entity_sims):.3f}')
if rated_e:
    scores_e = list(rated_e.values())
    print(f'  Manual rating (n={len(rated_e)}):   {np.mean(scores_e):.2f} / 5.0  (pass ≥3.5: {"YES" if np.mean(scores_e) >= 3.5 else "NO"})')
else:
    print(f'  Manual rating:          NOT YET RATED')
print()
print('FILES')
print(f'  /home/ishana/scratch/data/classes/paraphrase_classes.json')
print(f'  /home/ishana/scratch/data/classes/paraphrase_embeddings.npy')
print(f'  /home/ishana/scratch/data/classes/entity_classes.json')
print(f'  /home/ishana/scratch/data/classes/entity_embeddings.npy')
print(f'  /home/ishana/scratch/data/classes/similarity_histograms.png')
print(f'  /home/ishana/scratch/data/classes/entity_size_distribution.png')